<img width="20%" alt="EarthDaily Analytics" src="https://raw.githubusercontent.com/earthdaily/Images/main/Corporate/EarthDaily.png" style="border-radius: 15%">

# 📘 EarthDaily Agriculture - Crop id extraction class - Dev notebook

## **✅ Step 1: Initialisation**

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
import os
manager = WorkflowManager("prod", log_to_console=True, log_level="DEBUG")

## **🛠️ Step 2: Get entities**

### Option 1 - Load entities from Earthdaily platform

In [ ]:
manager.load_seasonfields()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 2 - Load entities from Earthdaily platform (Batch method - Recommended for large datasets)

In [ ]:
# This method uses batch processing to efficiently load 5000 entities

# Load all available entities using batch processing
manager.load_seasonfields_batch()

print("\n🔎 First entity loaded:")
print(manager.sfd_list[['id', 'name']].head())

### Option 3 -Load entities from file

In [ ]:
from earthdaily.agriculture.core.geometry import load_geodataframe

# Load entities from an external file (supports .shp, .parquet, .gpq, .geojson, .json, .gpkg, .csv)
file_path = "inputs/agroterrint.parquet"
manager.sfd_list = load_geodataframe(file_path, verbose=True)
print(manager.sfd_list.head())

## **📥 Step 3: Extract analytics - Debug function from processor_cropid_functions.py**

### 🗺️ Configure extraction

In [ ]:
# Import your class
from earthdaily.agriculture.extractors.cropid_functions import cropidExtractor
extractor = cropidExtractor(manager.bearer_token, manager.token_expiration,config=manager.config )

# Define column mapping to match DataFrame column names from the platform
column_mapping = {"crop": "crop.id"}

extractor.setup_cropid_parameters(
    begin_year=2020,
    end_year=2025,
    mask_type="EndSeason",
    limit_nb_crop=1,
    crop_mask_percent=75,
    mode="year",
    exclude_columns=[],
    column_mapping=column_mapping
)

### 🗺️ Test functions

In [ ]:
# Prepare test seasonfield_data
seasonfield_data = {
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop": "SOYBEANS"
}

#### Test get_cropid_api

In [ ]:
print("\n--- Test: get_cropid---")
# Invoke the function
try:
    result = extractor.get_cropid_api(seasonfield_data)
    print("✅ Raw API response received:")
    print(result if isinstance(result, dict) else result[:500])
except Exception as e:
    print(f"❌ API call failed: {e}")
    


#### Test get_cropid_api_safe

In [ ]:
print("\n--- Test: get_inseason_monitoring_safe ---")
safe_result = extractor.get_cropid_api_safe(seasonfield_data)
print(safe_result)

#### Test format_cropid_json

In [ ]:
print("\n--- Test: format_cropid_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_cropid_json(safe_result["data"])
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head())
else:
    print("⚠️ Skipping format_inseason_json: No valid data from API.")

#### Test format_cropid_year_json

In [ ]:
print("\n--- Test: format_cropid_year_json ---")
if safe_result["success"] and safe_result["data"]:
    formatted_df = extractor.format_cropid_year_json(safe_result["data"],seasonfield_data)
    print(f"✅ Formatted DataFrame with {len(formatted_df)} rows:")
    display(formatted_df.head())
else:
    print("⚠️ Skipping format_inseason_json: No valid data from API.")


#### Test format_cropid_historical_season_json

In [ ]:
print("\n--- Test: format_cropid_historical_season ---")
if safe_result["success"]:
    historical_seasons = extractor.format_cropid_historical_season_json(safe_result["data"], seasonfield_data)
    print(f"✅ Historical seasons: {historical_seasons}")
else:
    print("⚠️ Skipping: No valid data from API.")


#### Test format_cropid_full_history_json (wide-form pivot)

One row per entity, one column per year in `begin_year..end_year`, filled with the top crop code by `cropMaskPercent`. Missing years are `NaN`. Useful as an entity-level summary view to merge alongside other extractor outputs.


In [ ]:
print("--- Test: format_cropid_full_history_json ---")
if safe_result["success"]:
    full_history_df = extractor.format_cropid_full_history_json(safe_result["data"], seasonfield_data)
    print(f"✅ Full history shape: {full_history_df.shape}")
    print(f"   Year columns: {[c for c in full_history_df.columns if c != 'entity_id']}")
    display(full_history_df)
else:
    print("⚠️ Skipping: No valid data from API.")



### Review - all post-processing methods side-by-side

Re-runs each `format_*` method against the **same** API response (`safe_result['data']`) so you can compare what each mode produces from one extraction:

| Method | Mode | Shape |
|---|---|---|
| `format_cropid_json` | `history` | long-form, one row per (year, crop) |
| `format_cropid_year_json` | `year` | long-form, filtered to entity's crop |
| `format_cropid_historical_season_json` | `historical_season` | comma-separated string of years |
| `format_cropid_full_history_json` | `full_history` | wide-form, one row per entity, one column per year |


In [ ]:
print('=' * 70)
print('Side-by-side review of all four post-processing methods')
print('=' * 70)

if not (safe_result['success'] and safe_result['data']):
    print("Skipping review: no valid data in safe_result['data'].")
else:
    raw = safe_result['data']

    print()
    print('--- 1. format_cropid_json (mode="history") ---')
    history_df = extractor.format_cropid_json(raw, seasonfield_data)
    print(f'   shape: {history_df.shape}')
    display(history_df)

    print()
    print('--- 2. format_cropid_year_json (mode="year") ---')
    year_df = extractor.format_cropid_year_json(raw, seasonfield_data)
    print(f'   shape: {year_df.shape}')
    display(year_df)

    print()
    print('--- 3. format_cropid_historical_season_json (mode="historical_season") ---')
    seasons_str = extractor.format_cropid_historical_season_json(raw, seasonfield_data)
    print(f'   type: {type(seasons_str).__name__}')
    print(f'   value: {seasons_str!r}')

    print()
    print('--- 4. format_cropid_full_history_json (mode="full_history") ---')
    full_df = extractor.format_cropid_full_history_json(raw, seasonfield_data)
    print(f'   shape: {full_df.shape}')
    display(full_df)



#### Edge case - entity's crop is **not** in the API response

Builds a synthetic entity with a crop code that is unlikely to be present (`UNKNOWN_CROP_XYZ`) and re-runs each format method to confirm the documented empty-result behaviour:

- `format_cropid_year_json` -> empty DataFrame with `[entity_id, year, eda_crop_code]` schema
- `format_cropid_historical_season_json` -> empty string `""`
- `format_cropid_full_history_json` -> 1-row DataFrame with all year columns NaN (entity_id still set)
- `format_cropid_json` is unfiltered by entity crop, so it still returns the full long-form frame


In [ ]:
print('=' * 70)
print('Edge case - entity crop absent from API response')
print('=' * 70)

if not (safe_result['success'] and safe_result['data']):
    print("Skipping edge-case review: no valid data in safe_result['data'].")
else:
    raw = safe_result['data']
    # Synthetic entity, same id/geometry but a crop code that should not match.
    no_match_entity = dict(seasonfield_data)
    no_match_entity['crop'] = 'UNKNOWN_CROP_XYZ'

    print()
    print('--- year_json on absent crop ---')
    df_year = extractor.format_cropid_year_json(raw, no_match_entity)
    print(f'   empty: {df_year.empty}, columns: {list(df_year.columns)}')

    print()
    print('--- historical_season_json on absent crop ---')
    seasons = extractor.format_cropid_historical_season_json(raw, no_match_entity)
    print(f'   value: {seasons!r}')

    print()
    print('--- full_history_json on absent crop ---')
    df_full = extractor.format_cropid_full_history_json(raw, no_match_entity)
    print(f'   shape: {df_full.shape} - every year column should be NaN')
    display(df_full)

    print()
    print('--- format_cropid_json (unfiltered, still returns full history) ---')
    df_hist = extractor.format_cropid_json(raw, no_match_entity)
    print(f'   shape: {df_hist.shape}')
    display(df_hist.head())



#### `_matching_years` helper (used internally)

Both `year` and `historical_season` modes delegate to a shared static helper that returns the sorted unique list of years in which the API response reports a given `edaCropCode`. Direct demo:


In [ ]:
from earthdaily.agriculture.extractors.cropid_functions import cropidExtractor

if not (safe_result['success'] and safe_result['data']):
    print("Skipping helper demo: no valid data in safe_result['data'].")
else:
    raw = safe_result['data']
    entity_crop = seasonfield_data.get('crop')
    years = cropidExtractor._matching_years(raw, entity_crop)
    print(f'Years where edaCropCode == {entity_crop!r}: {years}')

    # Also try a couple of other codes that may or may not appear in the response.
    for code_str in ('SOYBEANS', 'CORN', 'COTTON', 'RICE'):
        print(f'  {code_str:>10s}: {cropidExtractor._matching_years(raw, code_str)}')


### 🗺️ process_single_entity

In [ ]:
import pandas as pd
row = pd.Series({
    "id": "z361x33",
    "geometry": "POLYGON ((-58.94540508 -13.72028589, -58.942163 -13.73172321, -58.928124090000004 -13.730314100000001, -58.93159922 -13.71888551, -58.94540508 -13.72028589))",
    "crop":"SOYBEANS"
})

result = extractor.process_single_entity_cropid(row)

print(result)

### 🗺️ process_inseason_bulk_extraction_parallel

In [ ]:
top25 = manager.sfd_list.head(10)
top25=top25.rename(columns={"crop.id": "crop"})
print(top25.columns)
# Launch extraction with 10 threads 

result = extractor.process_cropid_bulk_extraction_parallel(
    entity_list=manager.sfd_list,
    params=None,
    max_workers=20,
    output_path=manager.output_result_dir,
    fail_safe=False,
    #filter_column="",
    #filter_value="OTHERS",
    #filter_type="include" # filter type used to 'include' or 'exclude' row matching column and value filter
)

print(f"\nResults summary:")
print(f"Total: {result['total_calculations']}")
print(f"Success: {result['successful_calculations']}")
print(f"Failed: {result['failed_calculations']}")

print("\n🔍 First 3 errors:")
for i, error in enumerate(result['global_errors'][:3]):
    print(f"\nError {i+1}:")
    for key, value in error.items():
        print(f"  {key}: {value}")

In [ ]:
# Get the clean DataFrame
results=result["results_df"]
print(results.columns)

## Step 4: Workflow Integration Test

Test the CropID extractor in the pattern used by `WorkflowManager.run_workflow()`.
The workflow uses `mode="year"` to return `(entity_id, year, eda_crop_code)` per entity,
which downstream transforms consume to build image_id lists for zoning.

**YAML config reference** (`configuration/crop_history_zoning.yml`):
```yaml
steps:
  - name: cropid
    extractor: cropidExtractor
    module: earthdaily.agriculture.extractors.cropid_functions
    setup: { method: setup_cropid_parameters, params: { mode: year, begin_year: 2020, end_year: 2025 } }
    run: { method: process_cropid_bulk_extraction_parallel }
```

In [ ]:
# Workflow-style dynamic import & setup (mirrors WorkflowManager._execute_single_step)
import importlib

module_path = "earthdaily.agriculture.extractors.cropid_functions"
class_name = "cropidExtractor"

mod = importlib.import_module(module_path)
ExtractorClass = getattr(mod, class_name)

wf_extractor = ExtractorClass(manager.bearer_token, manager.token_expiration, config=manager.config)

# Setup with workflow parameters (mode: year for fertilizer zoning pipeline)
wf_extractor.setup_cropid_parameters(
    begin_year=2020,
    end_year=2025,
    mask_type="EndSeason",
    limit_nb_crop=1,
    crop_mask_percent=75,
    mode="year",
    column_mapping={"crop": "crop.id"},
)

print("Workflow-style extractor ready")

In [ ]:
# Workflow-style bulk extraction (mode=year)
# Returns entity_id, year, eda_crop_code — the format expected by downstream transforms
wf_result = wf_extractor.process_cropid_bulk_extraction_parallel(
    entity_list=manager.sfd_list.head(5),
    max_workers=5,
    skip_export=True,
    prefix="wf_cropid_year",
)

cropid_df = wf_result["results_df"]
print(f"Workflow output: {len(cropid_df)} rows")
print(f"Columns: {list(cropid_df.columns)}")
display(cropid_df.head(10))

# Verify expected columns for downstream transform
expected_cols = {"entity_id", "year", "eda_crop_code"}
actual_cols = set(cropid_df.columns)
if expected_cols.issubset(actual_cols):
    print("Downstream transform columns present")
else:
    missing = expected_cols - actual_cols
    print(f"Missing columns for downstream transform: {missing}")